# 1. Business Problem & Understanding

# 2. Libraries Import and Data Loading

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
import time
from tqdm import tqdm
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             roc_curve, precision_recall_curve, f1_score, recall_score, precision_score)

In [20]:
df = pd.read_csv("C:\Prima\Github\Data\loan_data_2007_2014.csv")
pd.set_option('display.max_columns', None)

C:\Users\prima\AppData\Local\Temp\ipykernel_29488\537107180.py:1: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:\Prima\Github\Data\loan_data_2007_2014.csv")


In [21]:
df.head()

,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_il_6m,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,B2,NaN,10+ years,RENT,24000.0,Verified,Dec-11,Fully Paid,n,https://www.lendingclub.com/browse/loanDetail....,Borrower added on 12/22/11 > I need to upgra...,credit_card,Computer,860xx,AZ,27.65,0.0,Jan-85,1.0,NaN,NaN,3.0,0.0,13648,83.7,9.0,f,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,Jan-15,171.62,NaN,Jan-16,0.0,NaN,1,INDIVIDUAL,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,C4,Ryder,< 1 year,RENT,30000.0,Source Verified,Dec-11,Charged Off,n,https://www.lendingclub.com/browse/loanDetail....,Borrower added on 12/22/11 > I plan to use t...,car,bike,309xx,GA,1.00,0.0,Apr-99,5.0,NaN,NaN,3.0,0.0,1687,9.4,4.0,f,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-13,119.66,NaN,Sep-13,0.0,NaN,1,INDIVIDUAL,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,RENT,12252.0,Not Verified,Dec-11,Fully Paid,n,https://www.lendingclub.com/browse/loanDetail....,NaN,small_business,real estate business,606xx,IL,8.72,0.0,Nov-01,2.0,NaN,NaN,2.0,0.0,2956,98.5,10.0,f,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,Jun-14,649.91,NaN,Jan-16,0.0,NaN,1,INDIVIDUAL,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,RENT,49200.0,Source Verified,Dec-11,Fully Paid,n,https://www.lendingclub.com/browse/loanDetail....,Borrower added on 12/21/11 > to pay for prop...,other,personel,917xx,CA,20.00,0.0,Feb-96,1.0,35.0,NaN,10.0,0.0,5598,21.0,37.0,f,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,Jan-15,357.48,NaN,Jan-15,0.0,NaN,1,INDIVIDUAL,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,B5,University Medical Group,1 year,RENT,80000.0,Source Verified,Dec-11,Current,n,https://www.lendingclub.com/browse/loanDetail....,Borrower added on 12/21/11 > I plan on combi...,other,Personal,972xx,OR,17.94,0.0,Jan-96,0.0,38.0,NaN,15.0,0.0,27783,53.9,38.0,f,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,Jan-16,67.79,Feb-16,Jan-16,0.0,NaN,1,INDIVIDUAL,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 75 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Unnamed: 0                   466285 non-null  int64  
 1   id                           466285 non-null  int64  
 2   member_id                    466285 non-null  int64  
 3   loan_amnt                    466285 non-null  int64  
 4   funded_amnt                  466285 non-null  int64  
 5   funded_amnt_inv              466285 non-null  float64
 6   term                         466285 non-null  object 
 7   int_rate                     466285 non-null  float64
 8   installment                  466285 non-null  float64
 9   grade                        466285 non-null  object 
 10  sub_grade                    466285 non-null  object 
 11  emp_title                    438697 non-null  object 
 12  emp_length                   445277 non-null  object 
 13 

### Dataset Columns Description

| Column Name | Description |
|------------|-------------|
| acc_now_delinq | Number of accounts on which the borrower is currently delinquent |
| addr_state | State provided by the borrower in the loan application |
| annual_inc | Self-reported annual income of the borrower |
| application_type | Indicates whether the loan is individual or joint application |
| collection_recovery_fee | Post charge-off collection fee |
| collections_12_mths_ex_med | Number of collections in the last 12 months excluding medical collections |
| delinq_2yrs | Number of 30+ days delinquency incidents in the past 2 years |
| desc | Loan description provided by the borrower |
| dti | Debt-to-income ratio excluding mortgage and requested loan |
| earliest_cr_line | Month when the borrower's earliest credit line was opened |
| emp_length | Employment length in years (0–10, where 10 means 10+ years) |
| emp_title | Job title provided by the borrower |
| funded_amnt | Total amount funded for the loan |
| funded_amnt_inv | Total amount funded by investors |
| grade | Loan grade assigned by Lending Club |
| home_ownership | Home ownership status (RENT, OWN, MORTGAGE, OTHER) |
| id | Unique loan listing ID |
| initial_list_status | Initial listing status of the loan (W, F) |
| inq_last_6mths | Number of credit inquiries in the past 6 months |
| installment | Monthly payment owed by the borrower |
| int_rate | Interest rate on the loan |
| issue_d | Month when the loan was issued |
| last_credit_pull_d | Most recent month credit was pulled |
| last_pymnt_amnt | Last payment amount received |
| last_pymnt_d | Month when the last payment was received |
| loan_amnt | Loan amount applied for by the borrower |
| loan_status | Current status of the loan (target variable) |
| member_id | Unique borrower ID |
| mths_since_last_delinq | Months since last delinquency |
| mths_since_last_major_derog | Months since most recent major derogatory event (90+ days late) |
| mths_since_last_record | Months since last public record |
| next_pymnt_d | Next scheduled payment date |
| open_acc | Number of open credit accounts |
| out_prncp | Remaining outstanding principal |
| out_prncp_inv | Remaining outstanding principal funded by investors |
| policy_code | Policy code (1 = publicly available, 2 = not publicly available) |
| pub_rec | Number of derogatory public records |
| purpose | Purpose of the loan |
| pymnt_plan | Indicates whether a payment plan is in place |
| recoveries | Post charge-off gross recovery |
| revol_bal | Total revolving credit balance |
| revol_util | Revolving credit utilization rate |
| sub_grade | Loan sub-grade assigned by Lending Club |
| term | Loan term in months (36 or 60) |
| title | Loan title provided by the borrower |
| tot_coll_amt | Total collection amounts ever owed |
| tot_cur_bal | Total current balance of all credit accounts |
| total_acc | Total number of credit accounts |
| total_pymnt | Total payments received to date |
| total_pymnt_inv | Total payments received by investors |
| total_rec_int | Total interest received |
| total_rec_late_fee | Total late fees received |
| total_rec_prncp | Total principal received |
| total_rev_hi_lim | Total revolving credit limit |
| url | URL of the loan listing page |
| verification_status | Income verification status |
| zip_code | First three digits of borrower’s ZIP code |

# 3. Exploratory Data Analysis & Data Cleaning

## 3.1 Drop Unnecessary Columns

In [23]:
check_missing = df.isnull().sum() * 100 / df.shape[0]
print("Here is columns with null values and its percentage:")
print(check_missing[check_missing > 0].sort_values(ascending=False))
print("="*42)
print(f'Total there are {len(check_missing[check_missing > 0])} columns with null values')

Here is columns with null values and its percentage:
inq_fi                         100.000000
open_rv_24m                    100.000000
max_bal_bc                     100.000000
all_util                       100.000000
inq_last_12m                   100.000000
annual_inc_joint               100.000000
verification_status_joint      100.000000
dti_joint                      100.000000
total_cu_tl                    100.000000
il_util                        100.000000
mths_since_rcnt_il             100.000000
total_bal_il                   100.000000
open_il_24m                    100.000000
open_il_12m                    100.000000
open_il_6m                     100.000000
open_acc_6m                    100.000000
open_rv_12m                    100.000000
mths_since_last_record          86.566585
mths_since_last_major_derog     78.773926
desc                            72.981975
mths_since_last_delinq          53.690554
next_pymnt_d                    48.728567
tot_cur_bal            

In [24]:
df.dropna(axis = 1, how = "all", inplace = True)

In [25]:
# 1. NO PREDICTIVE VALUE (identifiers, free text, duplicates)
no_predictive_value = [
    'Unnamed: 0', 'id', 'member_id', 'url',           # Identifiers
    'desc', 'title', 'emp_title',                      # Free text
    'sub_grade',                                        # Redundant with grade
    'funded_amnt', 'funded_amnt_inv',                  # Duplicate of loan_amnt
    'out_prncp', 'out_prncp_inv',                      # Post-loan info
    'zip_code',                                         # Too granular
]

# 2. DATA LEAKAGE (information not available at application time)
data_leakage = [
    'total_pymnt', 'total_pymnt_inv',                  # Payments made
    'total_rec_prncp', 'total_rec_int',                # Received payments
    'total_rec_late_fee',                              # Late fees incurred
    'recoveries', 'collection_recovery_fee',           # Post-default recovery
    'last_pymnt_d', 'last_pymnt_amnt',                 # Last payment info
    'next_pymnt_d',                                    # Future payment date
    'last_credit_pull_d',                              # Post-loan credit pull
    'pymnt_plan',                                      # Payment plan (post-issue)
    'acc_now_delinq',                                  # Current delinquency status
]

# 3. INSUFFICIENT VARIANCE (constant or extremely imbalanced)
insufficient_variance = [
    'policy_code',                                      # Constant value
    'initial_list_status',                              # Not information of a lender
]

# 4. COMBINE ALL COLUMNS TO DROP
useless_columns = list(set(
    no_predictive_value + 
    data_leakage + 
    insufficient_variance
))

In [26]:
df.drop(columns = useless_columns, inplace = True)

## 3.2 Drop Unnecessary Rows

In [27]:
df['loan_status'].value_counts(normalize=True)*100

loan_status
Current                                                48.087757
Fully Paid                                             39.619332
Charged Off                                             9.109236
Late (31-120 days)                                      1.479782
In Grace Period                                         0.674695
Does not meet the credit policy. Status:Fully Paid      0.426349
Late (16-30 days)                                       0.261214
Default                                                 0.178432
Does not meet the credit policy. Status:Charged Off     0.163205
Name: proportion, dtype: float64

In [28]:
loan_status_mapping = {
    # GOOD
    "Fully Paid": 0,
    "Does not meet the credit policy. Status:Fully Paid": 0,

    # BAD
    "Charged Off": 1,
    "Default": 1,
    "Does not meet the credit policy. Status:Charged Off": 1,
}

In [29]:
df = df[df['loan_status'].isin(loan_status_mapping)]
df["status_bad"] = df["loan_status"].map(loan_status_mapping)
df = df.dropna(subset=["status_bad"])
df["status_bad"] = df["status_bad"].astype("Int64")
df = df.drop(columns=["loan_status"])
df["status_bad"].value_counts(normalize=True)

status_bad
0    0.80906
1    0.19094
Name: proportion, dtype: Float64

## 3.3 Fix Data Type and Clean Its Values

In [30]:
check_missing = df.isnull().sum() * 100 / df.shape[0]
check_missing[check_missing > 0].sort_values(ascending = False)

mths_since_last_record         88.021404
mths_since_last_major_derog    82.568080
mths_since_last_delinq         56.190125
total_rev_hi_lim               28.802617
tot_cur_bal                    28.802617
tot_coll_amt                   28.802617
emp_length                      3.793410
revol_util                      0.097922
collections_12_mths_ex_med      0.062826
delinq_2yrs                     0.012565
earliest_cr_line                0.012565
inq_last_6mths                  0.012565
total_acc                       0.012565
open_acc                        0.012565
pub_rec                         0.012565
annual_inc                      0.001733
dtype: float64

In [31]:
# check data type of each column
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 230795 entries, 0 to 466283
Data columns (total 30 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    230795 non-null  int64  
 1   term                         230795 non-null  object 
 2   int_rate                     230795 non-null  float64
 3   installment                  230795 non-null  float64
 4   grade                        230795 non-null  object 
 5   emp_length                   222040 non-null  object 
 6   home_ownership               230795 non-null  object 
 7   annual_inc                   230791 non-null  float64
 8   verification_status          230795 non-null  object 
 9   issue_d                      230795 non-null  object 
 10  purpose                      230795 non-null  object 
 11  addr_state                   230795 non-null  object 
 12  dti                          230795 non-null  float64
 13  deli

In [32]:
# Checking basic data information
def check_data_information(df, cols):
    """
    This function provides detailed information about each column in a dataframe, including:
    - Data type of the column
    - Number of missing (null) values
    - Percentage of missing values
    - Total number of duplicated rows in the dataframe (not column-specific)
    - Number of unique values in the column
    - A sample of up to 5 unique values from the column

    Parameters:
    df (pd.DataFrame): The dataframe you want to check.
    cols (list): List of column names to check the information from.

    Returns:
    pd.DataFrame: A dataframe with detailed information for each column.
    """
    list_item = []
    for col in cols:
        list_item.append([col,                             # The column name
                          df[col].dtype,                   # The data type of the column
                          df[col].isna().sum(),            # The count of null values in the column
                          round(100 * df[col].isna().sum() / len(df[col]), 2),  # The percentage of null values
                          df.duplicated().sum(),           # The count of duplicated rows in the entire dataframe
                          df[col].nunique(),               # The count of unique values in the column
                          str(df[col].unique()[:5])])           # A sample of the first 5 unique values in the column

    desc_df = pd.DataFrame(data=list_item, columns='Feature, Data Type, Null Values, Null Percentage, Duplicated Values, Unique Values, Unique Sample'.split(","))
    return desc_df

In [33]:
columns_to_check = df.columns
check_data_information(df, columns_to_check)

,Feature,Data Type,Null Values,Null Percentage,Duplicated Values,Unique Values,Unique Sample
0,loan_amnt,int64,0,0.00,0,1308,[ 5000 2500 2400 10000 3000]
1,term,object,0,0.00,0,2,[' 36 months' ' 60 months']
2,int_rate,float64,0,0.00,0,505,[10.65 15.27 15.96 13.49 7.9 ]
3,installment,float64,0,0.00,0,43071,[162.87 59.83 84.33 339.31 156.46]
4,grade,object,0,0.00,0,7,['B' 'C' 'A' 'E' 'F']
5,emp_length,object,8755,3.79,0,11,['10+ years' '< 1 year' '3 years' '9 years' '4...
6,home_ownership,object,0,0.00,0,6,['RENT' 'OWN' 'MORTGAGE' 'OTHER' 'NONE']
7,annual_inc,float64,4,0.00,0,18195,[24000. 30000. 12252. 49200. 36000.]
8,verification_status,object,0,0.00,0,3,['Verified' 'Source Verified' 'Not Verified']
9,issue_d,object,0,0.00,0,91,['Dec-11' 'Nov-11' 'Oct-11' 'Sep-11' 'Aug-11']


- `application_type` only consist of single value. We'll drop later along with other column (if exist) after cleaning  
- Some object-typed columns consist many unique value. We'll also drop them along with object-typed column with few unique value but imbalance.

In [34]:
obj_col = df.select_dtypes(include = 'object').columns
obj_col

Index(['term', 'grade', 'emp_length', 'home_ownership', 'verification_status',
       'issue_d', 'purpose', 'addr_state', 'earliest_cr_line',
       'application_type'],
      dtype='object')

All correct as object except `earliest_cr_line` and `issue_d`  
Change `earliest_cr_line` and `issue_d` to date

### 3.3.1 Handle Datetime Type

In [35]:
def date_parse(val):
    if pd.isna(val):
        return pd.NaT
    
    month_str, year_str = val.split("-")
    year = int(year_str)
    
    if year >= 30:
        year += 1900
    else:
        year += 2000
    
    return pd.to_datetime(f"{year}-{month_str}-01", format="%Y-%b-%d")


In [36]:
df["earliest_cr_line"] = df["earliest_cr_line"].apply(date_parse)
df["issue_d"] = df["issue_d"].apply(date_parse)

# Check to make sure no date in which loan was issued is earlier than earliest credit line issued
(df["earliest_cr_line"] > df["issue_d"]).sum()

np.int64(0)

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 230795 entries, 0 to 466283
Data columns (total 30 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   loan_amnt                    230795 non-null  int64         
 1   term                         230795 non-null  object        
 2   int_rate                     230795 non-null  float64       
 3   installment                  230795 non-null  float64       
 4   grade                        230795 non-null  object        
 5   emp_length                   222040 non-null  object        
 6   home_ownership               230795 non-null  object        
 7   annual_inc                   230791 non-null  float64       
 8   verification_status          230795 non-null  object        
 9   issue_d                      230795 non-null  datetime64[ns]
 10  purpose                      230795 non-null  object        
 11  addr_state                   23

### 3.3.2 Handle Numeric Type

In [38]:
int_cols = df.select_dtypes(include='int64').columns
df[int_cols] = df[int_cols].astype('Int64')

float_cols = df.select_dtypes(include='float64').columns
df[float_cols] = df[float_cols].astype('Float64')

df['int_rate'] = df['int_rate']/100
df['dti'] = df['dti']/100
df['revol_util'] = df['revol_util']/100
df['revol_util'] = df['revol_util'].clip(upper=1.0)

In [39]:
df.duplicated().sum()

np.int64(0)

### 3.3.3 Handle Categorical Type

In [40]:
# Check each unique value of object-typed columns and their distribution
cat_col = df.select_dtypes(include = 'object').columns
for col in cat_col:
    print(f'Column name: ', col)
    print(df[col].unique())
    print(f"Distribution of unique value in column {col} (in percent):")
    print((df[col].value_counts(normalize = True)*100).sort_values(ascending = False))
    print("="*50)

Column name:  term
[' 36 months' ' 60 months']
Distribution of unique value in column term (in percent):
term
36 months    78.634286
60 months    21.365714
Name: proportion, dtype: float64
Column name:  grade
['B' 'C' 'A' 'E' 'F' 'D' 'G']
Distribution of unique value in column grade (in percent):
grade
B    30.599450
C    25.401763
A    16.945774
D    15.918889
E     7.342013
F     2.987933
G     0.804177
Name: proportion, dtype: float64
Column name:  emp_length
['10+ years' '< 1 year' '3 years' '9 years' '4 years' '5 years' '1 year'
 '6 years' '2 years' '7 years' '8 years' nan]
Distribution of unique value in column emp_length (in percent):
emp_length
10+ years    30.967844
2 years       9.749144
< 1 year      8.700685
3 years       8.449378
5 years       7.575662
1 year        7.014502
4 years       6.728517
6 years       6.235363
7 years       5.870113
8 years       4.818051
9 years       3.890740
Name: proportion, dtype: float64
Column name:  home_ownership
['RENT' 'OWN' 'MORTGAGE'

- `term` need to be stripped (left strip)
- Decrease `home_ownership`, `purpose`, `addr_state` unique values by merging rare values or grouping into larger categories
- Drop `application_type` because only consist of single value (drop later)

In [41]:
df['term'] = df['term'].str.strip()

In [42]:
# ============================================================================
# Group rare categories in home_ownership
# ============================================================================

# Create mapping dictionary
home_ownership_mapping = {
    'MORTGAGE': 'MORTGAGE',
    'RENT': 'RENT',
    'OWN': 'OWN',
    'OTHER': 'OTHER',
    'NONE': 'OTHER',
    'ANY': 'OTHER'
}

# Apply mapping
df['home_ownership_grouped'] = df['home_ownership'].map(home_ownership_mapping)

# Check the new distribution
print("="*60)
print("DISTRIBUTION AFTER GROUPING RARE CATEGORIES")
print("="*60)
grouped_dist = df['home_ownership_grouped'].value_counts()
grouped_pct = df['home_ownership_grouped'].value_counts(normalize=True) * 100

for category in grouped_dist.index:
    print(f"{category:20} : {grouped_dist[category]:8,} ({grouped_pct[category]:6.2f}%)")

# Drop original column and rename grouped column
df.drop('home_ownership', axis=1, inplace=True)
df.rename(columns={'home_ownership_grouped': 'home_ownership'}, inplace=True)

DISTRIBUTION AFTER GROUPING RARE CATEGORIES
MORTGAGE             :  113,627 ( 49.23%)
RENT                 :   97,606 ( 42.29%)
OWN                  :   19,334 (  8.38%)
OTHER                :      228 (  0.10%)


In [43]:
purpose_mapping = {
    # Debt related (combine debt_consolidation and credit_card)
    'debt_consolidation': 'debt_related',
    'credit_card': 'debt_related',
    
    # Home related (combine home_improvement, house, moving)
    'home_improvement': 'home_related',
    'house': 'home_related',
    'moving': 'home_related',
    
    # Major purchase (combine major_purchase and car)
    'major_purchase': 'major_purchase',
    'car': 'major_purchase',
    
    # Business & investment (combine small_business and renewable_energy)
    'small_business': 'business_investment',
    'renewable_energy': 'business_investment',
    
    # Life events (combine wedding, vacation, medical, educational)
    'wedding': 'life_events',
    'vacation': 'life_events',
    'medical': 'life_events',
    'educational': 'life_events',
    
    # Other (keep as is)
    'other': 'other'
}

# Apply mapping to create new grouped column
df['purpose_group'] = df['purpose'].map(purpose_mapping)

# Check if any purpose values were not mapped (should be none)
unmapped_values = df[df['purpose_group'].isna()]['purpose'].unique()
if len(unmapped_values) > 0:
    print(f"Warning: These purposes were not mapped: {unmapped_values}")
else:
    print("✓ All purpose values successfully mapped")

✓ All purpose values successfully mapped


In [44]:
df.drop(columns = 'purpose', inplace = True)

In [45]:
print("\n" + "="*60)
print("DISTRIBUTION OF GROUPED PURPOSE")
print("="*60)

# Get value counts and percentages
group_distribution = df['purpose_group'].value_counts()
group_percentages = df['purpose_group'].value_counts(normalize=True) * 100

# Display results
for group in group_distribution.index:
    print(f"{group:20} : {group_distribution[group]:8,} ({group_percentages[group]:5.2f}%)")



DISTRIBUTION OF GROUPED PURPOSE
debt_related         :  179,970 (77.98%)
home_related         :   16,794 ( 7.28%)
other                :   13,303 ( 5.76%)
major_purchase       :    9,312 ( 4.03%)
life_events          :    6,502 ( 2.82%)
business_investment  :    4,914 ( 2.13%)


In [46]:
# Display original unique states
print(f"Original unique states: {df['addr_state'].nunique()}")
print(f"Original state distribution:")
print(df['addr_state'].value_counts().head(10))

state_to_region = {
    'CT': 'Northeast', 'ME': 'Northeast', 'MA': 'Northeast', 'NH': 'Northeast',
    'RI': 'Northeast', 'VT': 'Northeast', 'NJ': 'Northeast', 'NY': 'Northeast',
    'PA': 'Northeast', 'IL': 'Midwest', 'IN': 'Midwest', 'MI': 'Midwest',
    'OH': 'Midwest', 'WI': 'Midwest', 'IA': 'Midwest', 'KS': 'Midwest',
    'MN': 'Midwest', 'MO': 'Midwest', 'NE': 'Midwest', 'ND': 'Midwest',
    'SD': 'Midwest', 'DE': 'South', 'FL': 'South', 'GA': 'South',
    'MD': 'South', 'NC': 'South', 'SC': 'South', 'VA': 'South', 'DC': 'South',
    'WV': 'South', 'AL': 'South', 'KY': 'South', 'MS': 'South', 'TN': 'South',
    'AR': 'South', 'LA': 'South', 'OK': 'South', 'TX': 'South', 'AZ': 'West',
    'CO': 'West', 'ID': 'West', 'MT': 'West', 'NV': 'West', 'NM': 'West',
    'UT': 'West', 'WY': 'West', 'AK': 'West', 'CA': 'West', 'HI': 'West',
    'OR': 'West', 'WA': 'West'
}

Original unique states: 50
Original state distribution:
addr_state
CA    39429
NY    19795
TX    17536
FL    16066
NJ     8931
IL     8534
PA     7752
GA     7317
VA     7245
OH     7066
Name: count, dtype: int64


In [47]:
df['addr_region'] = df['addr_state'].map(state_to_region)
df.drop('addr_state', axis=1, inplace=True)

In [48]:
print(df['addr_region'].value_counts(normalize = True))

addr_region
South        0.339639
West         0.294816
Northeast    0.208406
Midwest      0.157139
Name: proportion, dtype: float64


### 3.4.4 Drop Columns with Single Value (Constant) & High Base Rate (Imbalanced) Values

In [49]:
constant_columns = []

for col in df.columns:
    if col not in useless_columns:  # Avoid checking columns already marked for deletion
        try:
            if df[col].nunique() == 1:
                constant_columns.append(col)
                print(f"✓ Constant column found: '{col}' -> unique value: {df[col].unique()[0]}")
        except:
            pass

print(f"\nTotal constant columns found: {len(constant_columns)}")

✓ Constant column found: 'application_type' -> unique value: INDIVIDUAL

Total constant columns found: 1


In [50]:
imbalanced_columns = []

for col in df.columns:
    if col not in useless_columns and col not in constant_columns:
        try:
            # Get value counts as percentage
            value_counts = df[col].value_counts(normalize=True) * 100
            
            # Check if any category represents > 99% of data
            if value_counts.max() >= 99.0:
                imbalanced_columns.append(col)
                print(f"\n✓ Imbalanced column found: '{col}'")
                print(f"   Distribution: {value_counts.head(3).to_dict()}")
        except:
            pass

print(f"\nTotal imbalanced columns found: {len(imbalanced_columns)}")


✓ Imbalanced column found: 'collections_12_mths_ex_med'
   Distribution: {np.float64(0.0): 99.47062649035335, np.float64(1.0): 0.49685670929980486, np.float64(2.0): 0.027747669629308472}

Total imbalanced columns found: 1


In [51]:
columns_to_drop = constant_columns + imbalanced_columns
df.drop(columns = columns_to_drop, inplace=True)

## 3.4 Missing Value Handling

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 230795 entries, 0 to 466283
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   loan_amnt                    230795 non-null  Int64         
 1   term                         230795 non-null  object        
 2   int_rate                     230795 non-null  Float64       
 3   installment                  230795 non-null  Float64       
 4   grade                        230795 non-null  object        
 5   emp_length                   222040 non-null  object        
 6   annual_inc                   230791 non-null  Float64       
 7   verification_status          230795 non-null  object        
 8   issue_d                      230795 non-null  datetime64[ns]
 9   dti                          230795 non-null  Float64       
 10  delinq_2yrs                  230766 non-null  Float64       
 11  earliest_cr_line             23

In [53]:
df.head()

,loan_amnt,term,int_rate,installment,grade,emp_length,annual_inc,verification_status,issue_d,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,mths_since_last_major_derog,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,status_bad,home_ownership,purpose_group,addr_region
0,5000,36 months,0.1065,162.87,B,10+ years,24000.0,Verified,2011-12-01,0.2765,0.0,1985-01-01,1.0,<NA>,<NA>,3.0,0.0,13648,0.837,9.0,<NA>,<NA>,<NA>,<NA>,0,RENT,debt_related,West
1,2500,60 months,0.1527,59.83,C,< 1 year,30000.0,Source Verified,2011-12-01,0.01,0.0,1999-04-01,5.0,<NA>,<NA>,3.0,0.0,1687,0.094,4.0,<NA>,<NA>,<NA>,<NA>,1,RENT,major_purchase,South
2,2400,36 months,0.1596,84.33,C,10+ years,12252.0,Not Verified,2011-12-01,0.0872,0.0,2001-11-01,2.0,<NA>,<NA>,2.0,0.0,2956,0.985,10.0,<NA>,<NA>,<NA>,<NA>,0,RENT,business_investment,Midwest
3,10000,36 months,0.1349,339.31,C,10+ years,49200.0,Source Verified,2011-12-01,0.2,0.0,1996-02-01,1.0,35.0,<NA>,10.0,0.0,5598,0.21,37.0,<NA>,<NA>,<NA>,<NA>,0,RENT,other,West
5,5000,36 months,0.079,156.46,A,3 years,36000.0,Source Verified,2011-12-01,0.112,0.0,2004-11-01,3.0,<NA>,<NA>,9.0,0.0,7963,0.283,12.0,<NA>,<NA>,<NA>,<NA>,0,RENT,life_events,West


In [54]:
check_missing = df.isnull().sum() * 100 / df.shape[0]
check_missing[check_missing > 0].sort_values(ascending=False)

mths_since_last_record         88.021404
mths_since_last_major_derog    82.568080
mths_since_last_delinq         56.190125
tot_cur_bal                    28.802617
tot_coll_amt                   28.802617
total_rev_hi_lim               28.802617
emp_length                      3.793410
revol_util                      0.097922
earliest_cr_line                0.012565
delinq_2yrs                     0.012565
inq_last_6mths                  0.012565
open_acc                        0.012565
total_acc                       0.012565
pub_rec                         0.012565
annual_inc                      0.001733
dtype: float64

## Missing Values Analysis and Interpretation

Based on the dataset with **230,795 rows** (after filtering for good/bad loan classification), here is the analysis of columns containing missing values and their business interpretation.

### Missing Values Overview

| Column | Missing % | Interpretation of Missing |
|--------|-----------|---------------------------|
| `mths_since_last_record` | 88.02% | **NEVER had a public record** (bankruptcy, tax lien, judgment) |
| `mths_since_last_major_derog` | 82.57% | **NEVER had a major derogatory record** (charge-off, default) |
| `mths_since_last_delinq` | 56.19% | **NEVER been delinquent** (no 30+ day late payment) |
| `tot_cur_bal` | 28.80% | **No/thin credit history** - borrower may be new to credit system |
| `tot_coll_amt` | 28.80% | **No/thin credit history** - no collection data available |
| `total_rev_hi_lim` | 28.80% | **No/thin credit history** - no revolving credit limit |
| `emp_length` | 3.79% | **Not disclosed** - borrower chose not to provide or unemployed |
| `revol_util` | 0.10% | **No revolving credit** - borrower has no revolving accounts |
| `earliest_cr_line` | 0.01% | **Data inconsistency** - minimal impact (<60 rows) |
| `delinq_2yrs` | 0.01% | **Data inconsistency** - minimal impact (<60 rows) |
| `inq_last_6mths` | 0.01% | **Data inconsistency** - minimal impact (<60 rows) |
| `open_acc` | 0.01% | **Data inconsistency** - minimal impact (<60 rows) |
| `total_acc` | 0.01% | **Data inconsistency** - minimal impact (<60 rows) |
| `pub_rec` | 0.01% | **Data inconsistency** - minimal impact (<60 rows) |
| `annual_inc` | 0.002% | **Not disclosed** - only 8 rows affected |

---

### Detailed Interpretation

#### 1. Informative Missing Values (Missing = "Never Occurred")

These columns represent **positive credit signals** when missing. A null value indicates the borrower has a clean credit history with no negative events.

| Column | Business Meaning | Signal |
|--------|-----------------|--------|
| `mths_since_last_record` | Months since last public record | ✅ **POSITIVE** - No bankruptcies or liens |
| `mths_since_last_major_derog` | Months since last major derogatory | ✅ **POSITIVE** - No major defaults |
| `mths_since_last_delinq` | Months since last delinquency | ✅ **POSITIVE** - Always paid on time |

**Strategy:** Create binary flags to capture this information:
- `ever_public_record` (1 = has record, 0 = never)
- `ever_major_derog` (1 = has derogatory, 0 = never)
- `ever_delinquent` (1 = has delinquent, 0 = never)

Then fill original columns with `0` (meaning "0 months since occurrence" = never occurred).

#### 2. Thin Credit File (Missing = "No Credit History")

When `tot_cur_bal`, `tot_coll_amt`, and `total_rev_hi_lim` are all missing, it typically indicates a **thin credit file** - the borrower is new to the credit system or has very limited credit history.

| Column | Business Meaning |
|--------|-----------------|
| `tot_cur_bal` | Total current balance across all credit accounts |
| `tot_coll_amt` | Total collection amount |
| `total_rev_hi_lim` | Total revolving high credit limit |

**Strategy:** 
- Create a flag `has_credit_history` (1 = has credit data, 0 = thin file)
- Fill missing values with median values for modeling purposes

#### 3. Employment Information

**`emp_length` (3.79% missing)**
- Missing indicates borrower did **not disclose** employment information
- Could mean: unemployed, self-employed, freelancer, or chose not to answer
- **Strategy:** Fill with mode (most common employment length) or create an "Unknown" category

#### 4. Revolving Credit Information

**`revol_util` (0.10% missing)**
- Missing typically occurs when `revol_bal = 0` or `total_rev_hi_lim = 0`
- Indicates borrower has **no revolving credit** (no credit cards or unused lines)
- **Strategy:** Fill with `0` (zero utilization)

#### 5. Minor Missing Values (<0.02%)

The following columns have negligible missing values (less than 60 rows out of 230,795):
- `earliest_cr_line`
- `delinq_2yrs`
- `inq_last_6mths`
- `open_acc`
- `total_acc`
- `pub_rec`
- `annual_inc`

**Strategy:** Fill with median (numeric) or mode (categorical). Dropping these rows is also acceptable due to the tiny number affected.

---

### Handling Strategy Summary

```python
# Summary of handling approach
handling_strategy = {
    "Informative Missing (Never Occurred)": {
        "columns": ["mths_since_last_record", "mths_since_last_major_derog", "mths_since_last_delinq"],
        "action": "Create binary flag + fill with 0"
    },
    "Thin Credit File": {
        "columns": ["tot_cur_bal", "tot_coll_amt", "total_rev_hi_lim"],
        "action": "Create 'has_credit_history' flag + fill with median"
    },
    "Employment": {
        "columns": ["emp_length"],
        "action": "Fill with mode"
    },
    "No Revolving Credit": {
        "columns": ["revol_util"],
        "action": "Fill with 0"
    },
    "Minor Missing": {
        "columns": ["annual_inc", "delinq_2yrs", "inq_last_6mths", "open_acc", "total_acc", "pub_rec", "earliest_cr_line"],
        "action": "Fill with median/mode or drop rows"
    }
}
```

### Key Insight

**Missing values in credit data are often informative!** Unlike in many other domains, a null value frequently indicates a "never occurred" event, which is valuable predictive information. Simply removing rows with missing values would discard meaningful signals about borrower credit behavior, especially for columns with >50% missing rates.

Instead, we preserve this information by:
1. Creating binary flag columns to capture the "missing = never occurred" signal
2. Filling original columns with appropriate default values (0 for recency, median for balances)
3. Keeping all rows for modeling

This approach allows machine learning models to learn the risk differences between borrowers with clean history (missing = never) versus those with past issues (non-missing values).

In [55]:
# ============================================================================
# 1. CREATE FLAG COLUMNS (Informative Missing)
# ============================================================================

# 1a. mths_since_last_record -> ever_public_record
if 'mths_since_last_record' in df.columns:
    df['ever_public_record'] = df['mths_since_last_record'].notna().astype(int)
    # Note: fillna with 0 will be done AFTER split (involves statistic? No - 0 is constant)
    # But since 0 is a constant (not a statistic from data), we can do it here
    df['mths_since_last_record'] = df['mths_since_last_record'].fillna(0)
    print("✓ ever_public_record created")

# 1b. mths_since_last_major_derog -> ever_major_derog
if 'mths_since_last_major_derog' in df.columns:
    df['ever_major_derog'] = df['mths_since_last_major_derog'].notna().astype(int)
    df['mths_since_last_major_derog'] = df['mths_since_last_major_derog'].fillna(0)
    print("✓ ever_major_derog created")

# 1c. mths_since_last_delinq -> ever_delinquent
if 'mths_since_last_delinq' in df.columns:
    if 'ever_delinquent' not in df.columns:
        df['ever_delinquent'] = df['mths_since_last_delinq'].notna().astype(int)
    df['mths_since_last_delinq'] = df['mths_since_last_delinq'].fillna(0)
    print("✓ ever_delinquent created")

✓ ever_public_record created
✓ ever_major_derog created
✓ ever_delinquent created


In [56]:
check_missing = df.isnull().sum() * 100 / df.shape[0]
check_missing[check_missing > 0].sort_values(ascending=False)

total_rev_hi_lim    28.802617
tot_coll_amt        28.802617
tot_cur_bal         28.802617
emp_length           3.793410
revol_util           0.097922
pub_rec              0.012565
delinq_2yrs          0.012565
earliest_cr_line     0.012565
inq_last_6mths       0.012565
open_acc             0.012565
total_acc            0.012565
annual_inc           0.001733
dtype: float64

In [57]:
credit_columns = ['tot_cur_bal', 'tot_coll_amt', 'total_rev_hi_lim']

# Create pattern column (1 = missing, 0 = not missing)
df_missingpattern = (
    df[credit_columns].isnull().astype(int).astype(str).agg(''.join, axis=1)
)

# Count each pattern
pattern_counts = df_missingpattern.value_counts().sort_index()
pattern_percentages = df_missingpattern.value_counts(normalize=True).sort_index() * 100
print(pattern_counts)
print(pattern_percentages)
# df.drop(columns = 'missing_pattern', inplace = True)

000    164320
111     66475
Name: count, dtype: int64
000    71.197383
111    28.802617
Name: proportion, dtype: float64


In [58]:
# ============================================================================
# 2. CREDIT HISTORY FLAG (Not involving statistics - just checking notna)
# ============================================================================

df['has_credit_history'] = (
    df['tot_cur_bal'].notna() & 
    df['tot_coll_amt'].notna() & 
    df['total_rev_hi_lim'].notna()
).astype(int)
print("✓ has_credit_history created")

# Note
# Untuk ['tot_cur_bal', 'tot_coll_amt', 'total_rev_hi_lim'] karena pattern-nya 000 dan 111 saja, maka cukup dibuat 1
# flag column, yakni 'has_credit_history'

✓ has_credit_history created


In [59]:
# ============================================================================
# 3. EMP_LENGTH MISSING FLAG (Does not involve statistics)
# ============================================================================

if 'emp_length' in df.columns:
    df['emp_length_missing'] = df['emp_length'].isnull().astype(int)
    print("✓ emp_length_missing created")
    # Note: fillna with mode will be done AFTER split

✓ emp_length_missing created


In [60]:
# ============================================================================
# 4. REVOL_UTIL FILLNA WITH 0 (Constant value, not a statistic)
# ============================================================================

if 'revol_util' in df.columns:
    df['revol_util'] = df['revol_util'].fillna(0)
    print("✓ revol_util filled with 0")

✓ revol_util filled with 0


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 230795 entries, 0 to 466283
Data columns (total 33 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   loan_amnt                    230795 non-null  Int64         
 1   term                         230795 non-null  object        
 2   int_rate                     230795 non-null  Float64       
 3   installment                  230795 non-null  Float64       
 4   grade                        230795 non-null  object        
 5   emp_length                   222040 non-null  object        
 6   annual_inc                   230791 non-null  Float64       
 7   verification_status          230795 non-null  object        
 8   issue_d                      230795 non-null  datetime64[ns]
 9   dti                          230795 non-null  Float64       
 10  delinq_2yrs                  230766 non-null  Float64       
 11  earliest_cr_line             23

In [62]:
df.isnull().sum().sort_values(ascending=False)

total_rev_hi_lim               66475
tot_cur_bal                    66475
tot_coll_amt                   66475
emp_length                      8755
total_acc                         29
pub_rec                           29
inq_last_6mths                    29
earliest_cr_line                  29
open_acc                          29
delinq_2yrs                       29
annual_inc                         4
loan_amnt                          0
grade                              0
issue_d                            0
verification_status                0
installment                        0
term                               0
mths_since_last_delinq             0
mths_since_last_record             0
dti                                0
int_rate                           0
mths_since_last_major_derog        0
revol_util                         0
revol_bal                          0
status_bad                         0
home_ownership                     0
purpose_group                      0
a

In [63]:
X = df.drop('status_bad', axis=1)
y = df['status_bad']

# Split with stratification to handle imbalance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts(normalize=True)}")
print(f"y_test distribution:\n{y_test.value_counts(normalize=True)}")

X_train shape: (184636, 32)
X_test shape: (46159, 32)
y_train distribution:
status_bad
0    0.809062
1    0.190938
Name: proportion, dtype: Float64
y_test distribution:
status_bad
0    0.809051
1    0.190949
Name: proportion, dtype: Float64


In [64]:
# ============================================================================
# 1. FILL MISSING VALUES WITH STATISTICS FROM TRAINING SET
# ============================================================================

# 1a. Fill missing emp_length with mode from training
if 'emp_length' in X_train.columns:
    emp_mode = X_train['emp_length'].mode()[0]
    X_train['emp_length'] = X_train['emp_length'].fillna(emp_mode)
    X_test['emp_length'] = X_test['emp_length'].fillna(emp_mode)
    print(f"✓ emp_length filled with mode: '{emp_mode}'")

# 1b. Fill missing credit columns with median from training
credit_columns = ['tot_cur_bal', 'tot_coll_amt', 'total_rev_hi_lim']
for col in credit_columns:
    if col in X_train.columns:
        median_val = int(X_train[col].median())
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)
        print(f"✓ {col} filled with median: {median_val:,.0f}")

✓ emp_length filled with mode: '10+ years'
✓ tot_cur_bal filled with median: 80,383
✓ tot_coll_amt filled with median: 0
✓ total_rev_hi_lim filled with median: 22,100


In [65]:
# ============================================================================
# 2. CAPPING OUTLIERS WITH QUANTILES FROM TRAINING SET
# ============================================================================

# 2b. Cap other columns
cap_values = {}
col_with_outlier = ['annual_inc', 'tot_cur_bal', 'tot_coll_amt', 'total_rev_hi_lim', 'revol_bal']
for col in col_with_outlier:
    if col in X_train.columns:
        cap_value = int(X_train[col].quantile(0.99))
        cap_values[col] = cap_value
        X_train[col] = X_train[col].clip(upper=cap_value)
        X_test[col] = X_test[col].clip(upper=cap_value)
        print(f"✓ {col} capped at 99th percentile: {cap_value:,.0f}")

print("\n✓ All preprocessing AFTER split completed!")

✓ annual_inc capped at 99th percentile: 234,067
✓ tot_cur_bal capped at 99th percentile: 588,112
✓ tot_coll_amt capped at 99th percentile: 1,866
✓ total_rev_hi_lim capped at 99th percentile: 117,500
✓ revol_bal capped at 99th percentile: 80,954

✓ All preprocessing AFTER split completed!


In [66]:
# for set in [X_train, X_test]:
#     set.dropna(inplace = True)

train = X_train.copy()
train['status_bad'] = y_train

train = train.dropna()

X_train = train.drop(columns='status_bad')
y_train = train['status_bad']

test = X_test.copy()
test['status_bad'] = y_test

test = test.dropna()

X_test = test.drop(columns='status_bad')
y_test = test['status_bad']

In [67]:
# ============================================================================
# 5. FEATURE ENGINEERING (Tidak Melibatkan Statistik dari Data)
# ============================================================================
for set in [X_train, X_test]:
    # 5a. Credit history months (from date columns - deterministic)
    set['credit_history_months'] = (
        set['issue_d'].dt.to_period('M').astype(int) - 
        set['earliest_cr_line'].dt.to_period('M').astype(int)
    )
    set['credit_history_months'] = set['credit_history_months'].astype('Int64') # tambahkan clip(lower=0)
    print("✓ credit_history_months created")

    # 5b. Loan to income ratio (with constant clipping at 5.0 - not a statistic)
    set['loan_to_income_ratio'] = (set['loan_amnt'] / set['annual_inc']).clip(upper=5.0)
    print("✓ loan_to_income_ratio created")

    # 5c. Total negative events (aggregation of flags - deterministic)
    set['total_negative_events'] = (
        (set['delinq_2yrs'] > 0).astype(int) +
        (set['pub_rec'] > 0).astype(int) +
        set['ever_delinquent'] +
        set['ever_public_record'] +
        set['ever_major_derog']
    )
    print("✓ total_negative_events created")

    # 5d. Recent delinquency (deterministic rule)
    set['recent_delinquency'] = ((set['mths_since_last_delinq'] > 0) & 
                                (set['mths_since_last_delinq'] <= 12)).astype(int)
    print("✓ recent_delinquency created")

    # 5e. Average balance per account (with division by zero handling)
    set['avg_balance_per_account'] = (set['tot_cur_bal'] / set['open_acc'].replace(0, np.nan)).fillna(0)

✓ credit_history_months created
✓ loan_to_income_ratio created
✓ total_negative_events created
✓ recent_delinquency created
✓ credit_history_months created
✓ loan_to_income_ratio created
✓ total_negative_events created
✓ recent_delinquency created


In [68]:
# 2a. Cap avg_balance_per_account
cap_value = X_train['avg_balance_per_account'].quantile(0.99)
X_train['avg_balance_per_account'] = X_train['avg_balance_per_account'].clip(upper=cap_value)
X_test['avg_balance_per_account'] = X_test['avg_balance_per_account'].clip(upper=cap_value)
print(f"✓ avg_balance_per_account capped at 99th percentile: {cap_value:.2f}")

✓ avg_balance_per_account capped at 99th percentile: 58811.20


In [69]:
# ============================================================================
# FINAL VERIFICATION
# ============================================================================

print("\n" + "="*70)
print("FINAL MISSING VALUES CHECK")
print("="*70)
for set in [X_train, X_test]:
    remaining_missing = set.isnull().sum()
    remaining_missing = remaining_missing[remaining_missing > 0]

    if len(remaining_missing) > 0:
        print("Remaining missing values:")
        print(remaining_missing)
    else:
        print("✓ No missing values remaining! All columns have been handled.")

    print(f"\nTotal columns now: {set.shape[1]}")
    print(f"Total rows: {set.shape[0]:,}")


FINAL MISSING VALUES CHECK
✓ No missing values remaining! All columns have been handled.

Total columns now: 37
Total rows: 184,611
✓ No missing values remaining! All columns have been handled.

Total columns now: 37
Total rows: 46,155


In [ ]:
# ============================================================================
# 3. DROP COLUMNS NOT USED FOR MODELING
# ============================================================================

columns_to_drop = ['issue_d', 'earliest_cr_line']
X_train = X_train.drop(columns=[col for col in columns_to_drop if col in X_train.columns])
X_test = X_test.drop(columns=[col for col in columns_to_drop if col in X_test.columns])
print(f"\n✓ Dropped columns: {columns_to_drop}")


✓ Dropped columns: ['issue_d', 'earliest_cr_line']


# 3. Feature Engineering

In [ ]:
X_train.head(5)

,loan_amnt,term,int_rate,installment,grade,emp_length,annual_inc,verification_status,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,mths_since_last_major_derog,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,home_ownership,purpose_group,addr_region,ever_public_record,ever_major_derog,ever_delinquent,has_credit_history,emp_length_missing,credit_history_months,loan_to_income_ratio,total_negative_events,recent_delinquency,avg_balance_per_account
58566,18000,36 months,0.162,634.61,C,1 year,135000.0,Verified,0.2378,1.0,1.0,18.0,0.0,23.0,0.0,21103,0.392,40.0,0.0,0.0,297229.0,53900.0,MORTGAGE,debt_related,South,0,0,1,1,0,292,0.133333,2,0,12923.0
197349,9600,36 months,0.0603,292.19,A,10+ years,50000.0,Not Verified,0.1778,0.0,0.0,0.0,0.0,20.0,0.0,8623,0.154,58.0,0.0,0.0,212676.0,56100.0,MORTGAGE,debt_related,Northeast,0,0,0,1,0,248,0.192,0,0,10633.8
38717,2400,36 months,0.1071,78.25,B,4 years,36000.0,Verified,0.152,1.0,3.0,7.0,73.0,12.0,1.0,2823,0.332,17.0,0.0,0.0,80383.0,22100.0,MORTGAGE,debt_related,South,1,0,1,0,0,100,0.066667,4,1,6698.583333
32290,15000,36 months,0.1099,491.03,B,2 years,215000.0,Not Verified,0.0041,0.0,0.0,0.0,0.0,5.0,0.0,2583,0.159,12.0,0.0,0.0,80383.0,22100.0,MORTGAGE,home_related,South,0,0,0,0,0,151,0.069767,0,0,16076.6
197537,9600,36 months,0.1409,328.53,B,10+ years,47000.0,Not Verified,0.2129,0.0,1.0,0.0,0.0,6.0,0.0,10529,0.675,19.0,0.0,903.0,177810.0,15600.0,MORTGAGE,debt_related,West,0,0,0,1,0,201,0.204255,0,0,29635.0


In [ ]:
X_test.isnull().sum().sort_values(ascending = False)

loan_amnt                      0
term                           0
int_rate                       0
installment                    0
grade                          0
emp_length                     0
annual_inc                     0
verification_status            0
dti                            0
delinq_2yrs                    0
inq_last_6mths                 0
mths_since_last_delinq         0
mths_since_last_record         0
open_acc                       0
pub_rec                        0
revol_bal                      0
revol_util                     0
total_acc                      0
mths_since_last_major_derog    0
tot_coll_amt                   0
tot_cur_bal                    0
total_rev_hi_lim               0
home_ownership                 0
purpose_group                  0
addr_region                    0
ever_public_record             0
ever_major_derog               0
ever_delinquent                0
has_credit_history             0
emp_length_missing             0
credit_his

In [ ]:
for set in [X_train, X_test]:
    for col in set.columns:
        if set[col].dtype == "float64":
            s = set[col]
            is_integer_like = np.all(np.isclose(s.dropna() % 1, 0))
            if is_integer_like:
                set[col] = s.astype("Int64")

In [ ]:
for set in [X_train, X_test]:
    int_cols = set.select_dtypes(include='int64').columns
    set[int_cols] = set[int_cols].astype('Int64')

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 184611 entries, 58566 to 103734
Data columns (total 35 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    184611 non-null  Int64  
 1   term                         184611 non-null  object 
 2   int_rate                     184611 non-null  Float64
 3   installment                  184611 non-null  Float64
 4   grade                        184611 non-null  object 
 5   emp_length                   184611 non-null  object 
 6   annual_inc                   184611 non-null  Float64
 7   verification_status          184611 non-null  object 
 8   dti                          184611 non-null  Float64
 9   delinq_2yrs                  184611 non-null  Float64
 10  inq_last_6mths               184611 non-null  Float64
 11  mths_since_last_delinq       184611 non-null  Float64
 12  mths_since_last_record       184611 non-null  Float64
 13  

# 4. Data Split

In [ ]:
del set

In [ ]:
# ============================================================================
# TRANSFORM SPECIFIC COLUMNS FOR MODELING
# ============================================================================

print("\n" + "="*70)
print("TRANSFORMING COLUMNS FOR MODELING")
print("="*70)

# 4a. Transform 'term' column from '36 months' to 36
if 'term' in X_train.columns:
    X_train['term'] = X_train['term'].str.extract('(\d+)').astype('Int64')
    X_test['term'] = X_test['term'].str.extract('(\d+)').astype('Int64')
    print("✓ term transformed to numeric (36, 60)")

# 4b. Transform 'emp_length' using mapping dictionary
mapping_emp_length = {
    '< 1 year': 0.5,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}

if 'emp_length' in X_train.columns:
    X_train['emp_length'] = X_train['emp_length'].map(mapping_emp_length).astype('float64')
    X_test['emp_length'] = X_test['emp_length'].map(mapping_emp_length).astype('float64')
    print("✓ emp_length transformed to numeric (0.5 to 10)")

print(f"\nFinal X_train shape: {X_train.shape}")
print(f"Final X_test shape: {X_test.shape}")


TRANSFORMING COLUMNS FOR MODELING
✓ term transformed to numeric (36, 60)
✓ emp_length transformed to numeric (0.5 to 10)

Final X_train shape: (184611, 35)
Final X_test shape: (46155, 35)


In [ ]:
for set in [X_train, X_test]:
    int_cols = set.select_dtypes(include='float64').columns
    set[int_cols] = set[int_cols].astype('Float64')

del set

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 184611 entries, 58566 to 103734
Data columns (total 35 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    184611 non-null  Int64  
 1   term                         184611 non-null  Int64  
 2   int_rate                     184611 non-null  Float64
 3   installment                  184611 non-null  Float64
 4   grade                        184611 non-null  object 
 5   emp_length                   184611 non-null  Float64
 6   annual_inc                   184611 non-null  Float64
 7   verification_status          184611 non-null  object 
 8   dti                          184611 non-null  Float64
 9   delinq_2yrs                  184611 non-null  Float64
 10  inq_last_6mths               184611 non-null  Float64
 11  mths_since_last_delinq       184611 non-null  Float64
 12  mths_since_last_record       184611 non-null  Float64
 13  

In [ ]:
# ============================================================================
# 1. IDENTIFY COLUMN TYPES
# ============================================================================

# Get all columns except target
all_cols = X_train.columns.tolist()

# Identify column types
numeric_cols = X_train.select_dtypes(include=['Float64', 'Int64']).columns.tolist()

# Ordinal columns (have natural order)
ordinal_cols = ['grade']  # A, B, C, D, E, F, G (A is best, G is worst)

# Nominal columns (no natural order)
nominal_cols = ['term', 'verification_status', 'home_ownership', 'purpose_group', 'addr_region']

# Remove columns that are already in ordinal/nominal from numeric
numeric_cols = [col for col in numeric_cols if col not in ordinal_cols + nominal_cols]

print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols[:5]}...")
print(f"Ordinal columns ({len(ordinal_cols)}): {ordinal_cols}")
print(f"Nominal columns ({len(nominal_cols)}): {nominal_cols}")


Numeric columns (29): ['loan_amnt', 'int_rate', 'installment', 'emp_length', 'annual_inc']...
Ordinal columns (1): ['grade']
Nominal columns (5): ['term', 'verification_status', 'home_ownership', 'purpose_group', 'addr_region']


In [ ]:
# ============================================================================
# 2. DEFINE PREPROCESSING FOR EACH COLUMN TYPE
# ============================================================================

# Grade ordering (A is best, G is worst)
grade_categories = [['A', 'B', 'C', 'D', 'E', 'F', 'G']]

# Preprocessor for ordinal columns
ordinal_transformer = OrdinalEncoder(categories=grade_categories, handle_unknown='use_encoded_value', unknown_value=-1)

# Preprocessor for nominal columns
nominal_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Preprocessor for numeric columns
numeric_transformer = StandardScaler()

# Combine all preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('ord', ordinal_transformer, ordinal_cols),
        ('nom', nominal_transformer, nominal_cols)
    ],
    remainder='drop'  # Drop any columns not specified
)

print("\n✓ ColumnTransformer created successfully")


✓ ColumnTransformer created successfully


In [ ]:
# ============================================================================
# 3. CREATE FULL PIPELINE WITH XGBOOST
# ============================================================================

from sklearn.utils.class_weight import compute_class_weight

# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"Class weights: Good={weight_dict[0]:.2f}, Bad={weight_dict[1]:.2f}")

# Update XGBoost model with scale_pos_weight
# For binary classification, scale_pos_weight = weight_bad / weight_good
scale_pos_weight = weight_dict[1] / weight_dict[0]

# Create new pipeline with class weighting
xgb_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=scale_pos_weight  # This gives more weight to minority class
)

# Create pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

print("✓ Full pipeline created")

Class weights: Good=0.62, Bad=2.62
✓ Full pipeline created


In [66]:
# ============================================================================
# 4. HYPERPARAMETER TUNING WITH NESTED CROSS-VALIDATION
# ============================================================================

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV, StratifiedKFold, cross_val_score

print("\n" + "="*70)
print("HYPERPARAMETER TUNING WITH HALVINGGRIDSEARCHCV")
print("="*70)

# Parameter grid for XGBoost
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

# Outer CV for nested validation (3-fold for faster execution)
outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Create HalvingGridSearchCV for inner loop (much faster than GridSearchCV)
# HalvingGridSearchCV uses successive halving to eliminate bad parameter combinations early
halving_search = HalvingGridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,  # Inner CV folds
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    random_state=42,
    factor=3,  # Number of candidates to keep in each iteration
    min_resources='exhaust'  # Use all resources in final iteration
)

print("Performing Nested Cross-Validation with HalvingGridSearchCV...")
print("This is significantly faster than regular GridSearchCV")

# Nested CV scores (outer loop)
nested_scores = cross_val_score(
    halving_search, X_train, y_train,
    cv=outer_cv,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\nNested CV Results:")
print(f"  ROC-AUC scores: {nested_scores}")
print(f"  Mean ROC-AUC: {nested_scores.mean():.4f} (+/- {nested_scores.std() * 2:.4f})")


HYPERPARAMETER TUNING WITH HALVINGGRIDSEARCHCV
Performing Nested Cross-Validation with HalvingGridSearchCV...
This is significantly faster than regular GridSearchCV



Nested CV Results:
  ROC-AUC scores: [0.70644601 0.70535132 0.7088171 ]
  Mean ROC-AUC: 0.7069 (+/- 0.0029)


In [67]:
# ============================================================================
# 5. FIT FINAL MODEL WITH BEST PARAMETERS
# ============================================================================

print("\n" + "="*70)
print("FITTING FINAL MODEL ON FULL TRAINING SET")
print("="*70)

# Fit HalvingGridSearchCV on full training data to find best parameters
halving_search.fit(X_train, y_train)

# Get best model
best_model = halving_search.best_estimator_
best_params = halving_search.best_params_

print(f"\nBest parameters found:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

# Get best score from inner CV
print(f"\nBest cross-validation score: {halving_search.best_score_:.4f}")


FITTING FINAL MODEL ON FULL TRAINING SET
n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 5
min_resources_: 2279
max_resources_: 184611
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 108
n_resources: 2279
Fitting 3 folds for each of 108 candidates, totalling 324 fits
----------
iter: 1
n_candidates: 36
n_resources: 6837
Fitting 3 folds for each of 36 candidates, totalling 108 fits
----------
iter: 2
n_candidates: 12
n_resources: 20511
Fitting 3 folds for each of 12 candidates, totalling 36 fits
----------
iter: 3
n_candidates: 4
n_resources: 61533
Fitting 3 folds for each of 4 candidates, totalling 12 fits
----------
iter: 4
n_candidates: 2
n_resources: 184599
Fitting 3 folds for each of 2 candidates, totalling 6 fits


c:\Prima\Github\Lending_Company_Project\venv\lib\site-packages\xgboost\training.py:199: UserWarning: [13:23:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Best parameters found:
  classifier__colsample_bytree: 0.8
  classifier__learning_rate: 0.05
  classifier__max_depth: 3
  classifier__n_estimators: 200
  classifier__subsample: 0.8

Best cross-validation score: 0.7109


In [68]:
# ============================================================================
# 6. EVALUATE ON TEST SET (WEIGHTED)
# ============================================================================

from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             roc_curve, precision_recall_curve, f1_score, accuracy_score)

print("\n" + "="*70)
print("MODEL EVALUATION ON TEST SET")
print("="*70)

# Predictions
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\nPerformance Metrics:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"  F1-Score: {f1:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Good (0)', 'Bad (1)']))

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['Actual Good', 'Actual Bad'], columns=['Pred Good', 'Pred Bad'])
print(cm_df)


MODEL EVALUATION ON TEST SET

Performance Metrics:
  Accuracy: 0.6461
  ROC-AUC: 0.7102
  F1-Score: 0.4169

Classification Report:
              precision    recall  f1-score   support

    Good (0)       0.89      0.64      0.75     37341
     Bad (1)       0.30      0.66      0.42      8814

    accuracy                           0.65     46155
   macro avg       0.60      0.65      0.58     46155
weighted avg       0.78      0.65      0.68     46155


Confusion Matrix:
             Pred Good  Pred Bad
Actual Good      23985     13356
Actual Bad        2976      5838


In [69]:
# ============================================================================
# FIND OPTIMAL THRESHOLD FOR BUSINESS NEEDS
# ============================================================================

from sklearn.metrics import confusion_matrix, recall_score, precision_score

# Get probabilities
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Try different thresholds
thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

print("="*70)
print("THRESHOLD OPTIMIZATION")
print("="*70)
print(f"{'Threshold':<12} {'Recall (Bad)':<15} {'Precision (Bad)':<18} {'F1 (Bad)':<12}")
print("-"*70)

best_f1 = 0
best_threshold = 0.5

for thresh in thresholds:
    y_pred_thresh = (y_pred_proba >= thresh).astype(int)
    recall = recall_score(y_test, y_pred_thresh)
    precision = precision_score(y_test, y_pred_thresh)
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{thresh:<12.2f} {recall:<15.4f} {precision:<18.4f} {f1:<12.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh

print("-"*70)
print(f"\n✓ Best threshold: {best_threshold:.2f} (F1-Score: {best_f1:.4f})")

# Apply best threshold
y_pred_optimized = (y_pred_proba >= best_threshold).astype(int)

print("\n" + "="*70)
print("RESULTS WITH OPTIMAL THRESHOLD")
print("="*70)
print(f"Threshold: {best_threshold:.2f}")
print(f"\nConfusion Matrix:")
cm_optimized = confusion_matrix(y_test, y_pred_optimized)
print(pd.DataFrame(cm_optimized, index=['Actual Good', 'Actual Bad'], columns=['Pred Good', 'Pred Bad']))

print(f"\nClassification Report:")
cr_thresh = classification_report(y_test, y_pred_optimized, target_names=['Good (0)', 'Bad (1)'])
print(cr_thresh)

THRESHOLD OPTIMIZATION
Threshold    Recall (Bad)    Precision (Bad)    F1 (Bad)    
----------------------------------------------------------------------
0.10         0.9995          0.1922             0.3224      
0.15         0.9955          0.1973             0.3293      
0.20         0.9867          0.2041             0.3382      
0.25         0.9677          0.2122             0.3481      
0.30         0.9346          0.2247             0.3623      
0.35         0.8836          0.2417             0.3795      
0.40         0.8196          0.2590             0.3937      
0.45         0.7532          0.2824             0.4108      
0.50         0.6624          0.3042             0.4169      
0.55         0.5654          0.3327             0.4189      
0.60         0.4548          0.3639             0.4043      
0.65         0.3431          0.4022             0.3703      
0.70         0.2282          0.4364             0.2997      
----------------------------------------------------

In [70]:
# ============================================================================
# RETRAIN WITH SMOTE
# ============================================================================

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Apply preprocessing first to get numerical features
X_train_preprocessed = preprocessor.fit_transform(X_train)

# Apply SMOTE
smote = SMOTE(random_state=42, sampling_strategy=1)  # 100% minority after oversampling
X_train_smote, y_train_smote = smote.fit_resample(X_train_preprocessed, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE: {pd.Series(y_train_smote).value_counts().to_dict()}")

# Train XGBoost on SMOTE data
xgb_smote = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05
)

xgb_smote.fit(X_train_smote, y_train_smote)

# Evaluate on test set (preprocess test set first)
X_test_preprocessed = preprocessor.transform(X_test)
y_pred_smote = xgb_smote.predict(X_test_preprocessed)
y_pred_proba_smote = xgb_smote.predict_proba(X_test_preprocessed)[:, 1]

print("\n" + "="*70)
print("SMOTE MODEL RESULTS")
print("="*70)
print(classification_report(y_test, y_pred_smote, target_names=['Good (0)', 'Bad (1)']))

print("\nConfusion Matrix:")
cm_smote = confusion_matrix(y_test, y_pred_smote)
print(pd.DataFrame(cm_smote, index=['Actual Good', 'Actual Bad'], columns=['Pred Good', 'Pred Bad']))

Before SMOTE: {np.int64(0): 149360, np.int64(1): 35251}
After SMOTE: {np.int64(0): 149360, np.int64(1): 149360}


c:\Prima\Github\Lending_Company_Project\venv\lib\site-packages\xgboost\training.py:199: UserWarning: [13:23:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



SMOTE MODEL RESULTS
              precision    recall  f1-score   support

    Good (0)       0.83      0.96      0.89     37341
     Bad (1)       0.46      0.14      0.22      8814

    accuracy                           0.80     46155
   macro avg       0.64      0.55      0.55     46155
weighted avg       0.76      0.80      0.76     46155


Confusion Matrix:
             Pred Good  Pred Bad
Actual Good      35849      1492
Actual Bad        7546      1268


In [71]:
def calculate_granular_business_impact(df_test, y_true, y_pred, y_pred_proba=None):
    """
    Calculate business impact per loan (granular approach)
    
    Parameters:
    - df_test: Test dataframe with loan information
    - y_true: Actual labels (0 = good, 1 = bad)
    - y_pred: Predicted labels
    - y_pred_proba: Predicted probabilities (optional)
    
    Returns:
    - DataFrame with per-loan impact analysis
    - Summary of total business impact
    """
    
    # Create a copy for analysis
    df_analysis = df_test.copy()
    df_analysis['actual'] = y_true.values
    df_analysis['predicted'] = y_pred
    
    if y_pred_proba is not None:
        df_analysis['probability'] = y_pred_proba
    
    # ========================================================================
    # CALCULATE PROFIT/LOSS PER LOAN (INDIVIDUAL LEVEL)
    # ========================================================================
    
    # For good loans (actual = 0)
    # Profit = Total payments received - Loan amount
    # Or estimate from installment × term
    
    def calculate_loan_profit(row):
        """Calculate profit if this loan performs well"""
        if 'total_pymnt' in df_analysis.columns and pd.notna(row['total_pymnt']):
            # Use actual total payment if available
            return row['total_pymnt'] - row['loan_amnt']
        else:
            # Estimate from installment and term
            # Extract term in months
            term_num = row['term']
            #if 'term_months' not in df_analysis.columns:
            #    term_num = int(row['term'].split()[0]) if 'term' in df_analysis.columns else 36
            #else:
            #    term_num = row['term_months']
            estimated_total = row['installment'] * term_num
            return estimated_total - row['loan_amnt']
    
    def calculate_loan_loss(row):
        """Calculate loss if this loan defaults"""
        if 'recoveries' in df_analysis.columns and pd.notna(row['recoveries']):
            # Loss = Loan amount - recovered amount
            return row['loan_amnt'] - row['recoveries']
        else:
            # Loss = Full loan amount (conservative estimate)
            return row['loan_amnt']
    
    # Calculate profit and loss for each row
    df_analysis['profit_if_good'] = df_analysis.apply(calculate_loan_profit, axis=1)
    df_analysis['loss_if_bad'] = df_analysis.apply(calculate_loan_loss, axis=1)
    
    # ========================================================================
    # DETERMINE ACTUAL BUSINESS IMPACT BASED ON CONFUSION MATRIX POSITION
    # ========================================================================
    
    def get_business_impact(row):
        """
        Calculate actual financial impact based on prediction outcome
        """
        actual = row['actual']
        predicted = row['predicted']
        
        if actual == 0 and predicted == 0:
            # TN: Correctly approved good loan → GET PROFIT
            return row['profit_if_good'], 'TN - Profit from good loan'
        
        elif actual == 1 and predicted == 1:
            # TP: Correctly rejected bad loan → SAVED FROM LOSS
            return row['loss_if_bad'], 'TP - Saved from bad loan loss'
        
        elif actual == 1 and predicted == 0:
            # FN: Wrongly approved bad loan → ACTUAL LOSS
            return -row['loss_if_bad'], 'FN - Loss from bad loan approved'
        
        else:  # actual == 0 and predicted == 1
            # FP: Wrongly rejected good loan → OPPORTUNITY LOSS
            return -row['profit_if_good'], 'FP - Lost profit from good loan rejected'
    
    # Apply to each row
    df_analysis['business_impact'], df_analysis['impact_type'] = zip(*df_analysis.apply(get_business_impact, axis=1))
    
    # ========================================================================
    # AGGREGATE RESULTS
    # ========================================================================
    
    # Group by impact type
    impact_summary = df_analysis.groupby('impact_type')['business_impact'].agg(['sum', 'count']).reset_index()
    
    # Calculate totals
    total_tn_profit = df_analysis[df_analysis['impact_type'] == 'TN - Profit from good loan']['business_impact'].sum()
    total_tp_savings = df_analysis[df_analysis['impact_type'] == 'TP - Saved from bad loan loss']['business_impact'].sum()
    total_fn_loss = abs(df_analysis[df_analysis['impact_type'] == 'FN - Loss from bad loan approved']['business_impact'].sum())
    total_fp_loss = abs(df_analysis[df_analysis['impact_type'] == 'FP - Lost profit from good loan rejected']['business_impact'].sum())
    
    net_impact = total_tn_profit + total_tp_savings - total_fn_loss - total_fp_loss
    
    # ========================================================================
    # DISPLAY RESULTS
    # ========================================================================
    
    print("="*70)
    print("GRANULAR BUSINESS IMPACT ANALYSIS (PER LOAN)")
    print("="*70)
    
    print("\n📊 PER LOAN IMPACT BREAKDOWN:")
    print(f"\n{'Impact Type':<45} {'Total Impact':<20} {'# Loans':<10}")
    print("-"*70)
    
    for _, row in impact_summary.iterrows():
        impact_type = row['impact_type']
        total = row['sum']
        count = row['count']
        print(f"{impact_type:<45} ${total:>18,.2f} {count:>8,}")
    
    print("-"*70)
    print(f"{'TN Profit (Good loans approved)':<45} ${total_tn_profit:>18,.2f}")
    print(f"{'TP Savings (Bad loans rejected)':<45} ${total_tp_savings:>18,.2f}")
    print(f"{'FN Loss (Bad loans approved)':<45} -${total_fn_loss:>17,.2f}")
    print(f"{'FP Loss (Good loans rejected)':<45} -${total_fp_loss:>17,.2f}")
    print("="*70)
    print(f"{'NET BUSINESS IMPACT':<45} ${net_impact:>18,.2f}")
    print("="*70)
    
    # Calculate ROI
    total_loans_value = df_analysis['loan_amnt'].sum()
    roi = (net_impact / total_loans_value) * 100
    print(f"\n📈 ROI (Return on Loan Portfolio): {roi:.2f}%")
    
    # Average impact per loan
    print(f"💰 Average impact per loan: ${net_impact/len(df_analysis):,.2f}")
    
    return df_analysis, {
        'tn_profit': total_tn_profit,
        'tp_savings': total_tp_savings,
        'fn_loss': total_fn_loss,
        'fp_loss': total_fp_loss,
        'net_impact': net_impact,
        'roi': roi
    }

In [90]:
# ============================================================================
# RUN THE GRANULAR ANALYSIS
# ============================================================================

# After you have predictions on test set
df_test_with_predictions = X_test.copy()  # Your test features
#df_test_with_predictions['actual'] = y_test.values
#df_test_with_predictions['predicted'] = y_pred  # From your model

# Add loan amount and other needed columns
# Make sure these columns exist in your test set
required_cols = ['loan_amnt', 'installment', 'term']
for col in required_cols:
    if col not in df_test_with_predictions.columns:
        print(f"Warning: {col} not found in test set")

# Run granular analysis
print("="*40)
print("Granular Analysis for Weighted Model")
print("="*40)
df_impact_weighted, impact_summary_weighted = calculate_granular_business_impact(
    df_test=X_test.copy(),
    y_true=y_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba  # Optional
)
print("\n\n\n")
print("="*40)
print("Granular Analysis for Weighted + Threshold Model")
print("="*40)
df_impact_thresh, impact_summary_thresh = calculate_granular_business_impact(
    df_test=X_test.copy(),
    y_true=y_test,
    y_pred=y_pred_optimized
)
print("\n\n\n")
print("="*40)
print("Granular Analysis for SMOTE Model")
print("="*40)
df_impact_smote, impact_summary_smote = calculate_granular_business_impact(
    df_test=X_test.copy(),
    y_true=y_test,
    y_pred=y_pred_smote,
    y_pred_proba=y_pred_proba_smote
)

Granular Analysis for Weighted Model
GRANULAR BUSINESS IMPACT ANALYSIS (PER LOAN)

📊 PER LOAN IMPACT BREAKDOWN:

Impact Type                                   Total Impact         # Loans   
----------------------------------------------------------------------
FN - Loss from bad loan approved              $    -37,066,275.00    2,976
FP - Lost profit from good loan rejected      $    -80,761,563.24   13,356
TN - Profit from good loan                    $     56,658,760.08   23,985
TP - Saved from bad loan loss                 $     90,292,750.00    5,838
----------------------------------------------------------------------
TN Profit (Good loans approved)               $     56,658,760.08
TP Savings (Bad loans rejected)               $     90,292,750.00
FN Loss (Bad loans approved)                  -$    37,066,275.00
FP Loss (Good loans rejected)                 -$    80,761,563.24
NET BUSINESS IMPACT                           $     29,123,671.84

📈 ROI (Return on Loan Portfolio): 4.

In [73]:
# ============================================================================
# COMPARE DIFFERENT MODELS
# ============================================================================

print("\n" + "="*70)
print("COMPARING BUSINESS IMPACT ACROSS MODELS")
print("="*70)

# Example comparison (replace with your actual results)
models_comparison = pd.DataFrame({
    'Model': ['Weighted Model', 'Threshold Model', 'SMOTE Model'],
    'TN_Profit': [impact_summary_weighted['tn_profit'], impact_summary_thresh['tn_profit'], impact_summary_smote['tn_profit']],
    'TP_Savings': [impact_summary_weighted['tp_savings'], impact_summary_thresh['tp_savings'], impact_summary_smote['tp_savings']],
    'FN_Loss': [-impact_summary_weighted['fn_loss'], -impact_summary_thresh['fn_loss'], -impact_summary_smote['fn_loss']],
    'FP_Loss': [-impact_summary_weighted['fp_loss'], -impact_summary_thresh['fp_loss'], -impact_summary_smote['fp_loss']],
    'Net_Impact': [impact_summary_weighted['net_impact'], impact_summary_thresh['net_impact'], impact_summary_smote['net_impact']],
})

print(models_comparison.to_string(index=False))


COMPARING BUSINESS IMPACT ACROSS MODELS
          Model    TN_Profit  TP_Savings      FN_Loss      FP_Loss  Net_Impact
 Weighted Model  56658760.08  90292750.0  -37066275.0 -80761563.24 29123671.84
Threshold Model  69367978.08  79690150.0  -47668875.0 -68052345.24 33336907.84
    SMOTE Model 121513788.56  23896250.0 -103462775.0 -15906534.76 26040728.80


# Saving Final Preprocessing & Model Objects

In [74]:
# ============================================================================
# SAVE FINAL MODEL AND ALL PREPROCESSING OBJECTS
# ============================================================================

import joblib
import os
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Create models directory
os.makedirs('models', exist_ok=True)

print("="*70)
print("SAVING FINAL MODEL AND PREPROCESSING OBJECTS")
print("="*70)

SAVING FINAL MODEL AND PREPROCESSING OBJECTS


In [75]:
# ============================================================================
# 1. SAVE THE BEST MODEL (Threshold Model)
# ============================================================================

# Assuming 'best_model' is your trained threshold model
# best_model = your trained pipeline/model

joblib.dump(best_model, 'models/best_model.joblib')
print("✓ Saved: models/best_model.joblib")

✓ Saved: models/best_model.joblib


In [76]:
# ============================================================================
# 2. SAVE OPTIMAL THRESHOLD
# ============================================================================

OPTIMAL_THRESHOLD = 0.55  # Your optimal threshold from analysis
joblib.dump(OPTIMAL_THRESHOLD, 'models/optimal_threshold.joblib')
print(f"✓ Saved: models/optimal_threshold.joblib (threshold = {OPTIMAL_THRESHOLD})")

✓ Saved: models/optimal_threshold.joblib (threshold = 0.55)


In [77]:
joblib.dump(cap_value, 'models/avg_balance_cap.joblib')
print(f"✓ Saved: models/avg_balance_cap.joblib (avg_balance_cap = {cap_value})")

✓ Saved: models/avg_balance_cap.joblib (avg_balance_cap = 58811.2)


In [78]:
# ============================================================================
# 3. SAVE PREPROCESSOR (ColumnTransformer)
# ============================================================================

# If best_model is a pipeline, extract the preprocessor
if hasattr(best_model, 'named_steps') and 'preprocessor' in best_model.named_steps:
    preprocessor = best_model.named_steps['preprocessor']
    joblib.dump(preprocessor, 'models/preprocessor.joblib')
    print("✓ Saved: models/preprocessor.joblib")
else:
    print("⚠ Warning: Preprocessor not found in pipeline")

✓ Saved: models/preprocessor.joblib


In [79]:
# ============================================================================
# 4. SAVE COLUMN INFORMATION
# ============================================================================

column_info = {
    'numeric_cols': numeric_cols,
    'ordinal_cols': ordinal_cols,
    'nominal_cols': nominal_cols,
    'grade_categories': [['A', 'B', 'C', 'D', 'E', 'F', 'G']]
}
joblib.dump(column_info, 'models/column_info.joblib')
print("✓ Saved: models/column_info.joblib")

✓ Saved: models/column_info.joblib


In [80]:
# ============================================================================
# 5. SAVE EMP_LENGTH MAPPING
# ============================================================================

joblib.dump(mapping_emp_length, 'models/mapping_emp_length.joblib')
print("✓ Saved: models/mapping_emp_length.joblib")

✓ Saved: models/mapping_emp_length.joblib


In [81]:
joblib.dump(cap_values, 'models/cap_values.joblib')
print("✓ Saved: models/cap_values.joblib")

✓ Saved: models/cap_values.joblib


In [82]:
purpose_options = {
    "Debt Related": "debt_related",
    "Home Related": "home_related",
    "Major Purchase": "major_purchase",
    "Business Investment": "business_investment",
    "Life Events": "life_events",
    "Other": "other"
}

joblib.dump(purpose_options, 'models/purpose_options.joblib')
print("✓ Saved: models/purpose_options.joblib")

✓ Saved: models/purpose_options.joblib


In [96]:
home_ownership_options = {
    "Mortgage": "MORTGAGE",
    "Rent": "RENT",
    "Own": "OWN",
    "Other": "OTHER"
}

joblib.dump(home_ownership_options, 'models/home_ownership_options.joblib')
print("✓ Saved: models/home_ownership_options.joblib")

✓ Saved: models/home_ownership_options.joblib


In [83]:
# ============================================================================
# 6. SAVE FEATURE NAMES (for reference)
# ============================================================================

# Get feature names from preprocessor if available
try:
    if 'preprocessor' in locals():
        feature_names = preprocessor.get_feature_names_out()
        joblib.dump(feature_names, 'models/feature_names.joblib')
        print(f"✓ Saved: models/feature_names.joblib ({len(feature_names)} features)")
except:
    print("⚠ Warning: Could not extract feature names")

✓ Saved: models/feature_names.joblib (49 features)


In [84]:
business_impact = {
    'model': 'Threshold Model (Best Model)',
    'threshold': OPTIMAL_THRESHOLD,
    'tn_profit': impact_summary_thresh['tn_profit'],
    'tp_savings': impact_summary_thresh['tp_savings'],
    'fn_loss': impact_summary_thresh['fn_loss'],
    'fp_loss': impact_summary_thresh['fp_loss'],
    'net_impact': impact_summary_thresh['net_impact'],
    'roc_auc': roc_auc_score(y_test, y_pred_optimized),
    'recall_bad': recall_score(y_test, y_pred_optimized),
    'precision_bad': precision_score(y_test, y_pred_optimized),
    'saved_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}
joblib.dump(business_impact, 'models/business_impact.joblib')
print("✓ Saved: models/business_impact.joblib")

✓ Saved: models/business_impact.joblib


In [85]:
f1_score(y_test, y_pred_optimized)

0.4188626907073509

In [86]:
# ============================================================================
# 8. CREATE MODEL CARD (Documentation)
# ============================================================================

model_card = f"""
# Model Card: Loan Default Prediction

## Model Information
- **Model Name**: best_model
- **Model Type**: XGBoost Classifier with Threshold Tuning
- **Training Date**: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
- **Training Data**: LendingClub 2007-2014

## Hyperparameters
{best_model.named_steps['classifier'].get_params() if hasattr(best_model, 'named_steps') else 'N/A'}

## Decision Threshold
- **Optimal Threshold**: {OPTIMAL_THRESHOLD}
- **Selection Criteria**: Maximize net business impact

## Performance Metrics
| Metric | Value |
|--------|-------|
| ROC-AUC | {round(roc_auc_score(y_test, y_pred_optimized),2)} |
| Recall (Bad Loan) | {round(recall_score(y_test, y_pred_optimized),2)} |
| Precision (Bad Loan) | {round(precision_score(y_test, y_pred_optimized),2)} |
| F1-Score (Bad Loan) | {round(f1_score(y_test, y_pred_optimized),2)} |

## Business Impact
| Component | Amount |
|-----------|--------|
| TN Profit | ${round(impact_summary_thresh['tn_profit'],2)} |
| TP Savings | ${round(impact_summary_thresh['tp_savings'],2)} |
| FN Loss | ${round(impact_summary_thresh['fn_loss'],2)} |
| FP Loss | ${round(impact_summary_thresh['fn_loss'],2)} |
| **Net Impact** | **${round(impact_summary_thresh['fn_loss'],2)}** |

## Files Included
- best_model.joblib - The trained model
- optimal_threshold.joblib - Decision threshold
- preprocessor.joblib - ColumnTransformer for preprocessing
- column_info.joblib - Column type information
- mapping_emp_length.joblib - Employment length mapping
- business_impact.joblib - Business impact metrics

## Usage
```python
import joblib

# Load model and threshold
model = joblib.load('models/best_model.joblib')
threshold = joblib.load('models/optimal_threshold.joblib')

# Predict
probability = model.predict_proba(X)[0, 1]
prediction = 1 if probability >= threshold else 0
```
"""
with open('models/model_card.txt', 'w') as f:
    f.write(model_card)
print("✓ Saved: models/model_card.txt")

print("\n" + "="*70)
print("✅ ALL OBJECTS SAVED SUCCESSFULLY!")
print("="*70)
print("\nFiles saved in 'models/' directory:")
for file in os.listdir('models'):
    print(f"  - {file}")

✓ Saved: models/model_card.txt

✅ ALL OBJECTS SAVED SUCCESSFULLY!

Files saved in 'models/' directory:
  - avg_balance_cap.joblib
  - best_model.joblib
  - business_impact.joblib
  - cap_values.joblib
  - column_info.joblib
  - feature_names.joblib
  - mapping_emp_length.joblib
  - model_card.txt
  - optimal_threshold.joblib
  - preprocessor.joblib
  - purpose_options.joblib
  - sample_test_data.csv


In [87]:
# Daftar visualisasi untuk mencari insight

# Univariate Analysis
## Target variable (barplot)                                      ()
## Kolom-kolom kategorik dengan nilai unik yang sedikit           ()
## Kolom-kolom numerik
### loan_amnt - kdeplot
### annual_inc - kdeplot
### dti - box-plot
## Kolom dengan missing value tinggi                              ()
## Kolom yang berhubungan langsung dengan pertanyaan bisnis       ()

# Bivariate Analysis

# Multivariate Analysis

# Note:
## - Visualisasi hanya dilakukan pada data training, setelah train test split
## - Apapun yang dilakukan pada data training juga diterapkan pada data testing

In [94]:
X_train['home_ownership'].value_counts()

home_ownership
MORTGAGE    90860
RENT        78195
OWN         15376
OTHER         180
Name: count, dtype: int64